In [1]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from pathlib import Path
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
DATA_DIR   = Path("raw_data")  # Input CSVs directory
ART_DIR    = Path("artifacts") # Output artifacts directory
ART_DIR.mkdir(exist_ok=True)

print("--- Starting Data Preprocessing ---")

# 1) Load & lowercase
print("Loading people.csv...")
people = pd.read_csv(
    DATA_DIR / "people.csv",
    low_memory=False,
    dtype={"first_name": str, "middle_name": str, "patronym": str, "surname": str}
)
people.columns = people.columns.str.lower().str.strip()
people = people.set_index("id")

# 2) Canonicalize names & flags
print("Canonicalizing names and flags...")
NAME_COLS = ["first_name", "middle_name", "patronym", "surname"]
for col in NAME_COLS:
    if col not in people.columns:
        people[col] = ""
people["full_name"] = (
    people[NAME_COLS]
    .fillna("")
    .apply(lambda r: " ".join(w.strip().lower() for w in r if w), axis=1)
)
people["birthyear"] = pd.to_numeric(people["birthyear"], errors="coerce").fillna(0).astype(int)
people["heimild"]   = pd.to_numeric(people["heimild"],   errors="coerce").fillna(0).astype(int)
people["sex_male"]  = people["sex"].apply(lambda x: 1 if isinstance(x, str) and x.lower()=="karl" else 0).astype(int)
people["has_partner"]= people["partner"].notna().astype(int)
people["has_father"] = people["father"].notna().astype(int)
people["has_mother"] = people["mother"].notna().astype(int)

# 3) Load and join manntol IDs
print("Loading manntol_einstaklingar_new.csv and merging ID columns…")
mann = pd.read_csv(
    DATA_DIR / "manntol_einstaklingar_new.csv",
    dtype=str,
    usecols=["id","bi_sokn","bi_hreppur","bi_sysla","thsk_maki","thsk_fadir","thsk_modir"]
).rename(columns={
    "bi_sokn":    "parish_id",
    "bi_hreppur": "district_id",
    "bi_sysla":   "county_id",
    "thsk_maki":  "partner_mann",
    "thsk_fadir": "father_mann",
    "thsk_modir": "mother_mann",
})
mann["id"] = mann["id"].astype(people.index.dtype)
mann = mann.set_index("id")
people = people.join(
    mann[["parish_id","district_id","county_id","partner_mann","father_mann","mother_mann"]],
    how="left"
)
people["partner"] = people["partner"].fillna(people["partner_mann"])
people["father"]  = people["father"].fillna(people["father_mann"])
people["mother"]  = people["mother"].fillna(people["mother_mann"])
people = people.drop(columns=["partner_mann","father_mann","mother_mann"])

# 4) Merge geography
for fname, idcol, merge_on, newcol in [
    ("parishes.csv",  "id", "parish_id",   "parish_full"),
    ("districts.csv", "id", "district_id", "district_name"),
    ("counties.csv",  "id", "county_id",   "county_name"),
]:
    p = DATA_DIR / fname
    if not p.exists(): continue
    geo = (
        pd.read_csv(p, low_memory=False)
          .rename(columns=str.lower)
          .astype({idcol:str})
    )
    geo_col = geo.set_index(idcol)[ "full_name" if newcol=="parish_full" else "name" ] \
                 .rename(newcol)
    people = people.join(geo_col, on=merge_on)


# 5) Impute static attributes
print("Imputing static attributes…")
if "person" in people.columns:
    for col in ["full_name","birthyear","sex_male"]:
        if col in people.columns:
            people[col] = people.groupby("person")[col].transform(lambda x: x.ffill().bfill())
else:
    print("Warning: 'person' column not found. Skipping imputation.")
people["full_name"] = people["full_name"].fillna("unknown")
people["birthyear"] = people["birthyear"].fillna(0).astype(int)
people["sex_male"] = people["sex_male"].fillna(0).astype(int)

# 6) **Include ALL rows** for ML features and graph
print("Including ALL rows for ML features and graph…")
people_ml = people.copy()
print(f"→ {len(people_ml)} total rows")

# Export full row_labels.csv
pd.DataFrame({
    "row_id": people_ml.index,
    "person": people_ml["person"].astype(str)
}).to_csv(ART_DIR / "row_labels.csv", index=False)

# Export just the linked subset
people_ml.loc[people_ml["person"].notna(), ["person"]] \
         .to_csv(ART_DIR / "rows_with_person.csv", index_label="row_id")

# 7) Define feature sets
print("Defining feature sets…")
NUM_COLS = ["birthyear","heimild","sex_male","has_partner","has_father","has_mother"]
CAT_LOW  = ["status","marriagestatus","district_name","county_name"]
CAT_HIGH = ["parish_full"]
for col in NUM_COLS+CAT_LOW+CAT_HIGH:
    if col not in people_ml.columns:
        people_ml[col] = 0 if col in NUM_COLS else ""

# 8) Numeric matrix
print("Creating numeric features…")
X_num = people_ml[NUM_COLS].values.astype(np.float32)

# 9) One-hot low-cardinality
print("One-hot encoding low-cardinality…")
ohe   = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
X_low = ohe.fit_transform(people_ml[CAT_LOW].fillna("").astype(str))
low_cols = ohe.get_feature_names_out(CAT_LOW)

# 10) Ordinal high-cardinality
print("Ordinal encoding high-cardinality…")
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_high = ord_enc.fit_transform(people_ml[CAT_HIGH].fillna("").astype(str)).astype(np.float32)

# 11) Save sparse features
print("Saving sparse features...")
sparse.save_npz(
    ART_DIR / "iceid_ml_ready.npz",
    sparse.hstack([sparse.csr_matrix(X_num), X_low, sparse.csr_matrix(X_high)], format="csr")
)

# 12) Create temporal graph
print("Creating temporal graph…")
row_id_to_idx = {rid:i for i,rid in enumerate(people_ml.index)}
edges = []
for _,grp in people_ml[people_ml["person"].notna()].groupby("person"):
    ids = grp.sort_values("heimild").index
    idx = [row_id_to_idx[r] for r in ids]
    edges += [[u,v] for u,v in zip(idx, idx[1:])] + [[v,u] for u,v in zip(idx, idx[1:])]
edge_index = (torch.tensor(edges, dtype=torch.long).t()
              if edges else torch.empty((2,0),dtype=torch.long))
graph = Data(edge_index=edge_index)
graph.node_id = torch.tensor(people_ml.index.values, dtype=torch.long)
# save only the *structure* (edges + node IDs), not x
torch.save({
    "edge_index": graph.edge_index,
    "node_id":    graph.node_id
}, ART_DIR / "temporal_graph.pt")
print(f"Graph: {graph.num_nodes} nodes, {graph.num_edges//2} undirected edges.")
print("--- Data Preprocessing Finished ---")

/home/potatosalad/anaconda3/envs/mscthesis/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Starting Data Preprocessing ---
Loading people.csv...
Canonicalizing names and flags...
Loading manntol_einstaklingar_new.csv and merging ID columns…
Imputing static attributes…
Including ALL rows for ML features and graph…
→ 984028 total rows
Defining feature sets…
Creating numeric features…
One-hot encoding low-cardinality…
Ordinal encoding high-cardinality…
Saving sparse features...
Creating temporal graph…
Graph: 984028 nodes, 266817 undirected edges.
--- Data Preprocessing Finished ---


In [2]:
#!/usr/bin/env python3
import numpy as np
from scipy import sparse
from pathlib import Path
import json
import pandas as pd

# Configuration
ART_DIR = Path("artifacts")
DATA_DIR = Path("raw_data")

# Load metadata
print("Loading metadata from preprocessed files...")
# Feature matrix shape from .npz
X = sparse.load_npz(ART_DIR / "iceid_ml_ready.npz")
n_nodes, n_feats = X.shape

# Number of records from people.csv
people = pd.read_csv(DATA_DIR / "people.csv", low_memory=False)
n_records = len(people)

# Generate Croissant metadata
print("Generating Croissant metadata...")
croissant = {
    "@context": {
        "schema": "http://schema.org/",
        "ml": "http://mlcommons.org/croissant/"
    },
    "@type": "schema:Dataset",
    "name": "IceID Entity Resolution Dataset",
    "description": "A dataset of Icelandic population records for entity resolution, processed into a sparse feature matrix with person ID labels.",
    "license": "CC BY 4.0",
    "distribution": [
        {
            "@type": "schema:DataDownload",
            "name": "feature_matrix",
            "contentUrl": "iceid_ml_ready.npz",
            "encodingFormat": "application/x-npz",
            "description": f"Sparse CSR matrix of shape ({n_nodes}, {n_feats}) containing node features in compressed format."
        },
        {
            "@type": "schema:DataDownload",
            "name": "row_labels",
            "contentUrl": "row_labels.csv",
            "encodingFormat": "text/csv",
            "description": f"CSV file with {n_nodes} rows, containing columns 'row_id' (original record IDs) and 'person' (person IDs) for entity resolution labels."
        },
        {
            "@type": "schema:DataDownload",
            "name": "raw_people",
            "contentUrl": "smb_data/people.csv",
            "encodingFormat": "text/csv",
            "description": f"Raw CSV file with {n_records} records, containing columns like 'id', 'heimild', 'first_name', 'middle_name', 'patronym', 'surname', 'birthyear', used for additional features like full_name and census year."
        }
    ],
    "recordSet": [
        {
            "@type": "schema:RecordSet",
            "name": "records",
            "description": f"Set of {n_nodes} records representing individual entries for entity resolution.",
            "field": [
                {
                    "@type": "schema:Field",
                    "name": "row_id",
                    "description": "Integer ID corresponding to the row index in the feature matrix and 'id' in people.csv.",
                    "dataTypes": "schema:Integer",
                    "source": {
                        "distribution": "row_labels",
                        "extract": {
                            "column": "row_id"
                        }
                    }
                },
                {
                    "@type": "schema:Field",
                    "name": "features",
                    "description": f"Sparse vector of {n_feats} float32 values, including numerical features (birthyear, heimild, sex_male, has_partner, has_father, has_mother), one-hot encoded categorical features (status, marriagestatus, district_name, county_name), and ordinal encoded parish_full.",
                    "dataTypes": "schema:QuantitativeValue",
                    "source": {
                        "distribution": "feature_matrix"
                    }
                },
                {
                    "@type": "schema:Field",
                    "name": "person",
                    "description": "Person ID (integer) linking records of the same individual, or -1 if unknown.",
                    "dataTypes": "schema:Integer",
                    "source": {
                        "distribution": "row_labels",
                        "extract": {
                            "column": "person"
                        }
                    }
                },
                {
                    "@type": "schema:Field",
                    "name": "full_name",
                    "description": "Concatenated name string derived from first_name, middle_name, patronym, and surname.",
                    "dataTypes": "schema:Text",
                    "source": {
                        "distribution": "raw_people",
                        "extract": {
                            "column": "first_name, middle_name, patronym, surname",
                            "transform": "join with spaces"
                        }
                    }
                },
                {
                    "@type": "schema:Field",
                    "name": "heimild",
                    "description": "Census year (integer) for the record.",
                    "dataTypes": "schema:Integer",
                    "source": {
                        "distribution": "raw_people",
                        "extract": {
                            "column": "heimild"
                        }
                    }
                }
            ]
        }
    ],
    "description": f"""
    A dataset for entity resolution derived from Icelandic population records. Contains {n_nodes} records with sparse feature vectors and person ID labels.

    **Loading Instructions:**
    - **Feature Matrix**: Load with `X = scipy.sparse.load_npz('artifacts/iceid_ml_ready.npz')`. Shape is ({n_nodes}, {n_feats}). Convert to dense with `X.toarray()` if needed.
    - **Labels**: Load with `df = pd.read_csv('artifacts/row_labels.csv')`. Columns are 'row_id' (record ID) and 'person' (person ID, -1 if unknown).
    - **Raw Data**: Load with `people = pd.read_csv('smb_data/people.csv')`. Use 'id' to align with row_labels.csv, and derive 'full_name' by concatenating 'first_name', 'middle_name', 'patronym', 'surname'. 'heimild' provides census year.
    """
}

# Save Croissant metadata
with open(ART_DIR / "croissant.json", "w") as f:
    json.dump(croissant, f, indent=2)
print("Croissant metadata file 'croissant.json' has been generated in the artifacts directory.")

Loading metadata from preprocessed files...
Generating Croissant metadata...
Croissant metadata file 'croissant.json' has been generated in the artifacts directory.
